In [ ]:
import os
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://www.opensanctions.org"
HEADERS = {"User-Agent": "Mozilla/5.0 (study project)"}
COUNTRY = "Albania"
EXPECTED = 141          # /pep/ 표 기준 예상 인원
NEWS_LIMIT = None       # 테스트할 땐 10 정도로 (None이면 전원 조회, 약 12분)

# ───────────────────────────────────────────────
# 1단계: /pep/ 페이지에서 알바니아 데이터셋 찾기
# ───────────────────────────────────────────────
res = requests.get(f"{BASE}/pep/", headers=HEADERS, timeout=10)
print("상태 코드:", res.status_code)
if res.status_code != 200:
    raise SystemExit("페이지를 못 받았어요 (차단 또는 오류).")
if COUNTRY not in res.text:
    raise SystemExit(f"HTML 소스에 '{COUNTRY}'가 없어요 → 동적 페이지일 가능성.")

soup = BeautifulSoup(res.text, "html.parser")

sources = {}
for a in soup.find_all("a", href=True):
    href, text = a["href"], a.get_text(strip=True)
    if href.startswith("/datasets/") and COUNTRY in text:
        slug = href.strip("/").split("/")[-1]
        sources[slug] = text

print(f"\n{COUNTRY} 관련 데이터셋 {len(sources)}개")
for slug, name in sources.items():
    print(f" - {name} ({slug})")
if not sources:
    raise SystemExit("데이터셋 링크를 못 찾았어요.")

# ───────────────────────────────────────────────
# 1-2단계: 데이터셋마다 CSV 받아서 합치기
# ───────────────────────────────────────────────
frames = []
for slug, name in sources.items():
    time.sleep(10)                                   # Crawl-delay 준수
    ds_url = f"{BASE}/datasets/{slug}/"
    ds_soup = BeautifulSoup(requests.get(ds_url, headers=HEADERS, timeout=10).text, "html.parser")
    csv_links = [urljoin(ds_url, a["href"]) for a in ds_soup.find_all("a", href=True) if ".csv" in a["href"]]
    csv_url = csv_links[0] if csv_links else \
        f"https://data.opensanctions.org/datasets/latest/{slug}/targets.simple.csv"
    try:
        df_src = pd.read_csv(csv_url)
        df_src["source"] = name
        frames.append(df_src)
        print(f"✅ {name}: {len(df_src)}행")
    except Exception as e:
        print(f"⚠ {name}: 다운로드 실패 ({e})")

if not frames:
    raise SystemExit("수집된 데이터가 없어요.")

all_people = pd.concat(frames, ignore_index=True)
if "schema" in all_people.columns:                   # 사람만 남기기
    all_people = all_people[all_people["schema"] == "Person"]
all_people = all_people.drop_duplicates("id").reset_index(drop=True)
print(f"\n인물 {len(all_people)}명 (예상 약 {EXPECTED}명)")

# ───────────────────────────────────────────────
# 2단계: 제재명단 파일과 대조
# ───────────────────────────────────────────────
SANC_URL = "https://data.opensanctions.org/datasets/latest/sanctions/targets.simple.csv"
SANC_FILE = "sanctions_cache.csv"
keep = {"id", "schema", "name", "birth_date", "countries", "sanctions", "dataset"}

if os.path.exists(SANC_FILE):                        # 한 번 받으면 재사용
    sanc = pd.read_csv(SANC_FILE, low_memory=False)
else:
    print("\n제재명단 다운로드 중... (용량이 커서 시간이 걸려요)")
    sanc = pd.read_csv(SANC_URL, usecols=lambda c: c in keep, low_memory=False)
    sanc.to_csv(SANC_FILE, index=False)
print(f"제재명단 {len(sanc):,}건")

# 2-1) ID 일치: 같은 인물로 확정된 경우
id_hits = all_people.merge(sanc, on="id", how="inner", suffixes=("", "_sanc"))

# 2-2) 이름 일치: 동명이인 가능 → 후보
norm = lambda s: " ".join(str(s).lower().split())
sanc_persons = sanc[sanc.get("schema", "Person") == "Person"] if "schema" in sanc else sanc
sanc_names = set(sanc_persons["name"].map(norm))
all_people["name_norm"] = all_people["name"].map(norm)
name_hits = all_people[all_people["name_norm"].isin(sanc_names) & ~all_people["id"].isin(id_hits["id"])]

print(f"\n[제재명단 ID 일치] {len(id_hits)}명")
if len(id_hits):
    print(id_hits[["name", "birth_date", "sanctions", "dataset_sanc"]].to_string(index=False))
print(f"\n[이름만 일치하는 후보 — 생년월일로 직접 확인] {len(name_hits)}명")
if len(name_hits):
    print(name_hits[["name", "birth_date"]].to_string(index=False))

# ───────────────────────────────────────────────
# 3단계: GDELT 뉴스 API로 최근 기사 수 + 헤드라인
# ───────────────────────────────────────────────
GDELT = "https://api.gdeltproject.org/api/v2/doc/doc"

def get_news(name, timespan="3months", maxrecords=75):
    params = {"query": f'"{name}"', "mode": "ArtList", "format": "json",
              "maxrecords": maxrecords, "timespan": timespan, "sort": "DateDesc"}
    try:
        r = requests.get(GDELT, params=params, headers=HEADERS, timeout=30)
        return r.json().get("articles", [])
    except Exception:
        return []                                    # 응답이 JSON이 아니면 빈 결과

targets = all_people["name"].tolist()[:NEWS_LIMIT]
news_rows, counts = [], []
for i, name in enumerate(targets, 1):
    arts = get_news(name)
    counts.append({"name": name, "article_count": len(arts)})
    for art in arts:
        news_rows.append({"name": name, "date": art.get("seendate", "")[:8],
                          "title": art.get("title"), "domain": art.get("domain"),
                          "language": art.get("language"), "url": art.get("url")})
    print(f"[{i}/{len(targets)}] {name}: {len(arts)}건")
    time.sleep(5)                                    # GDELT 권장 간격

count_df = pd.DataFrame(counts).sort_values("article_count", ascending=False)
news_df = pd.DataFrame(news_rows)
print(f"\n기사 총 {len(news_df)}건 (0건인 인물 {sum(count_df['article_count'] == 0)}명)")

TOP_N = 10
print(f"\n[최근 3개월 언론 노출 상위 {TOP_N}명]")
print(count_df.head(TOP_N).to_string(index=False))

for name in count_df.head(TOP_N)["name"]:
    print(f"\n■ {name}")
    for _, row in news_df[news_df["name"] == name].head(5).iterrows():
        print(f"  {row['date']} | {row['title']} ({row['domain']})")

# ───────────────────────────────────────────────
# 4단계: 결과 저장 (분석용, 재배포 금지)
# ───────────────────────────────────────────────
id_hits.to_csv("albania_sanction_id_hits.csv", index=False, encoding="utf-8-sig")
name_hits.to_csv("albania_sanction_name_candidates.csv", index=False, encoding="utf-8-sig")
count_df.to_csv("albania_news_counts.csv", index=False, encoding="utf-8-sig")
news_df.to_csv("albania_news_headlines.csv", index=False, encoding="utf-8-sig")
print("\n✅ 저장 완료")         # 한 줄의 HTML 구조

상태 코드: 200

Albania 관련 데이터셋 1개
 - Albania Members of Parliament (al_kuvendi)
✅ Albania Members of Parliament: 140행

인물 140명 (예상 약 141명)

제재명단 다운로드 중... (용량이 커서 시간이 걸려요)
제재명단 73,810건

[제재명단 ID 일치] 0명

[이름만 일치하는 후보 — 생년월일로 직접 확인] 0명
[1/140] Delina Sokol Ibrahimaj: 0건
[2/140] Bora Kastriot Muzhaqi: 0건
[3/140] Gent Hysen Strazimiri: 0건
[4/140] Romina Argjir Kuko: 0건
[5/140] Arbjan Cvetko Mazniku: 0건
[6/140] Greta SHYQYRI Bardeli: 0건
[7/140] Erald Izja Kapri: 0건
[8/140] Tedi Gaqo Blushi: 0건
[9/140] Agron NIKOLLË Gjekmarkaj: 0건
[10/140] Blendi Agim Gonxhja: 0건
[11/140] Tomor FEJZI Alizoti: 0건
[12/140] Adea Luan Pirdeni: 0건
[13/140] Blendi PËLLUMB Klosi: 0건
[14/140] Bujar FARUDIN Leskaj: 0건
[15/140] Eduard PAJTIM Shalsi: 0건
[16/140] Erion MIRDASH Braçe: 0건
[17/140] Pirro Thoma Vengu: 0건
[18/140] Niko ALEKS Peleshi: 0건
[19/140] Olta ANXHELO Xhaçka: 0건
[20/140] Taulant KASEM Balla: 0건
[21/140] Tërmet Nuro Peçi: 0건
[22/140] Albana Xhemal Koçiu: 0건
[23/140] Ervin Bashkim Hoxha: 0건
[24/140] Redi F

알바니아 PEP 제재 스크리닝 및 뉴스 헤드라인 수집 리포트
1. 목표 질문

특정 국가(알바니아)의 PEP 명단을 발췌한 뒤, 두 가지 질문에 답하는 것을 목표로 했다. 첫째, 명단의 인물 중 제재 대상이 있는가. 둘째, 각 인물에 관한 뉴스 헤드라인을 자동으로 찾을 수 있는가. 이는 금융기관 KYC 절차의 제재 스크리닝과 부정적 언론 검색(adverse media screening)을 축소해 재현한 것이다.

2. 데이터와 방법

PEP 명단은 OpenSanctions의 /pep/ 페이지에서 알바니아 데이터셋을 찾아 공식 CSV로 받았다. 이 페이지는 HTML 소스에 데이터가 들어 있는 정적(서버 렌더링) 페이지임을 확인했으므로 requests + BeautifulSoup을 사용했고, 브라우저 자동화 도구(Selenium, Playwright)는 필요하지 않다고 판단했다.

제재 여부는 OpenSanctions가 제공하는 제재명단 통합 파일과 대조했다. 처음에는 인물 페이지를 하나씩 조회하려 했으나, robots.txt가 /search와 /entities 경로를 금지하고 있어 공식 파일 대조로 방법을 바꿨다. 대조는 두 가지 기준으로 했다. OpenSanctions가 같은 인물에 부여하는 공통 ID가 일치하면 "확정", 이름만 일치하면 동명이인 가능성이 있으므로 "후보"로 분류했다.

뉴스는 구글 뉴스 자동 검색 대신 GDELT 뉴스 API를 사용했다. 구글 검색 자동화는 약관 위반이며 CAPTCHA에 막히는 반면, GDELT는 전 세계 뉴스를 수집해 연구용 API로 공개하므로 "공식 API 우선" 원칙에 맞는다. 인물별로 최근 3개월 기사 수와 헤드라인을 수집하도록 했다.

3. 수행 과정과 결과
3.1 명단 발췌

/pep/ 페이지에서 알바니아 관련 데이터셋은 1개(Albania Members of Parliament, al_kuvendi)였다. 상태 코드 200으로 정상 수신했고, CSV에서 인물 140명을 확보했다. 표에 표시된 엔티티 수(141)와 1건 차이가 나는데, 인물이 아닌 항목이 섞여 있었거나 업데이트 시점 차이로 보이며 결과에 미치는 영향은 작다고 판단했다.

3.2 질문 1: 제재 대상인가
항목	결과
제재명단 전체 건수	73,810건
ID 일치 (확정)	0명
이름 일치 (후보)	0명

ID 일치 0명은 비교적 신뢰할 수 있는 결과다. OpenSanctions가 의원 명단과 제재명단에서 동일 인물로 연결해 둔 사례가 없다는 뜻이기 때문이다.

반면 이름 일치 0명은 그대로 믿기 어렵다고 판단했다. 명단을 살펴보니 이름이 “Delina Sokol Ibrahimaj”, “Greta SHYQYRI Bardeli”처럼 세 단어로 되어 있었다. 알바니아 공식 명단은 이름과 성 사이에 아버지 이름(부칭) 을 넣는 형식인 반면, 제재명단은 보통 이름과 성만 쓰기 때문에 정확 일치 방식으로는 걸리지 않을 수 있다. 또한 ë, ç 같은 알바니아어 특수문자 표기 차이도 매칭을 방해할 수 있다.

3.3 질문 2: 뉴스 헤드라인을 찾을 수 있는가

첫 실행에서 140명 중 처음 21명을 조회한 결과, 전원 기사 0건이었다. 현직 국회의원 전원이 3개월간 보도가 전혀 없을 가능성은 낮으므로, 이를 "조용한 실패"(에러 없이 잘못된 결과가 나오는 상황)로 판단하고 실행을 중단했다.

원인으로 두 가지를 확인했다. 첫째, 3.2와 같은 이름 형식 문제다. 검색어가 "Delina Sokol Ibrahimaj"처럼 부칭이 포함된 정확한 구문이었는데, 실제 기사는 “Delina Ibrahimaj”로 쓰기 때문에 일치하는 기사가 없었다. 둘째, 코드가 API 요청 실패와 "기사 없음"을 모두 0건으로 처리하고 있어 실패를 구분할 수 없는 구조였다.

이에 따라 코드를 수정했다. 이름을 이름+성으로 줄이고, 특수문자가 있는 표기와 없는 표기를 함께 검색하며, 요청 실패는 별도로 표시하도록 했다. 같은 방식으로 제재명단 이름 대조도 다시 수행한다.

4. 재실행 결과 (수정 후 채울 칸)
항목	결과
API 정상 여부 테스트 (“Edi Rama”)	[ ]건
제재 후보 (이름+성 기준)	[ ]명
뉴스 조회 성공 / 실패	[ ]명 / [ ]명
기사 1건 이상인 인물	[ ]명
수집된 기사 총수	[ ]건
언론 노출 상위 인물	[ ]

상위 인물의 대표 헤드라인: [ ]

5. 목표 질문에 대한 답 (현재까지)

질문 1 (제재 대상인가). OpenSanctions가 동일 인물로 연결한 기준(ID)으로는 알바니아 의원 140명 중 제재 대상은 0명이다. 다만 이름 기준 대조는 형식 문제로 재확인이 필요하며, 재실행에서 후보가 나오면 생년월일을 대조해 최종 판단한다. 또한 이번 대조는 "제재(sanctions)" 데이터만 포함했으므로, 부패 관련 입국 금지나 거래 제한처럼 다른 분류에 속한 위험 정보는 범위 밖이다.

질문 2 (뉴스 헤드라인을 찾을 수 있는가). 방법상으로는 가능하다. GDELT API로 인물별 기사와 헤드라인을 자동 수집하는 구조를 만들었다. 그러나 1차 실행은 이름 형식 불일치로 실패했고, 원인을 찾아 수정했다. 최종 답은 4절의 재실행 결과로 확정한다.

6. 한계

GDELT는 최근 3개월만 다루며, 인물당 최대 75건까지만 수집했다. 흔한 이름은 다른 인물의 기사가 섞일 수 있고, 기사 수는 노출량일 뿐 부정적 보도인지는 헤드라인을 직접 읽어 판단해야 한다. PEP 분류와 제재 후보 여부는 그 자체로 위법을 뜻하지 않는다.

7. 윤리 점검

robots.txt를 확인해 허용 경로(/pep/, /datasets/)만 접근했고, 금지 경로(/search, /entities)는 공식 파일로 대체했다. 요청 간격은 OpenSanctions 10초(Crawl-delay), GDELT 5초(권장 간격)를 지켰다. 데이터는 비상업 학습용(CC BY-NC)으로만 사용하며 원본 명단은 재배포하지 않는다.

8. 배운 점

결과가 “0"으로 나왔을 때 그대로 결론을 내리지 않고 의심한 것이 이번 분석의 핵심이었다. 공식 명단과 외부 데이터의 이름 표기 형식이 다르면 수집 도구가 아무리 정확해도 매칭은 실패한다. 정확도는 더 강한 수집 도구가 아니라 데이터 형식을 이해하고 검증하는 과정에서 나온다는 것을 확인했다.